In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

In [2]:
checkpoint = "mistralai/Mistral-7B-v0.3"
quantization_config = BitsAndBytesConfig(load_in_8bit=True)
mistral_tokenizer = AutoTokenizer.from_pretrained(checkpoint)
mistral_model = AutoModelForCausalLM.from_pretrained(checkpoint, device_map="auto", quantization_config=quantization_config)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [3]:
def generate(model, tokenizer, prompt, max_new_tokens=50, **generate_kwargs):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, pad_token_id=tokenizer.eos_token_id, **generate_kwargs)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [4]:
prompt = "It often comes to my mind, to be or not to"
print(generate(mistral_model, mistral_tokenizer, prompt))

It often comes to my mind, to be or not to be.

I’m not sure if I’m a good person. I’m not sure if I’m a bad person. I’m not sure if I’m a good person. I’m not sure if I


In [5]:
prompt = "It often comes to my mind, to be or not to"
print(generate(mistral_model, mistral_tokenizer, prompt, do_sample=True));

It often comes to my mind, to be or not to be a vegetarian. Should I ditch my meat-eating habit? I eat meat because I like it. Not because I have been told to eat it. I have been told to eat all kinds of vegetables because it is good for health


In [6]:
prompt = "It often comes to my mind, to be or not to"
print(generate(mistral_model, mistral_tokenizer, prompt, do_sample=True, top_p=0.8, temperature=0.9));

It often comes to my mind, to be or not to be, that is the question. The question I am referring to is, of course, “Am I doing the right thing?” I have asked myself this question on more than one occasion. I have asked this question in regards to my work, my


the above temperature and top_p params seem to be reasonably good...

In [7]:
prompt = "I've got a good feeling about Arsenal this season. Boy I do feel something"
print(generate(mistral_model, mistral_tokenizer, prompt, do_sample=True, top_p=0.8, temperature=0.9));

I've got a good feeling about Arsenal this season. Boy I do feel something good in the air.

You might have read some of my previous articles on this site. If not, just know I'm a 100% positive person and I believe in the club 100%. I believe in


Using mistral for answering questions - using in-context learning

In [16]:
DEFAULT_TEMPLATE = "Capital city of France = Paris \nCapital city of {country} = "
prompt = DEFAULT_TEMPLATE.format(country="Wakanda")
ans = generate(mistral_model, mistral_tokenizer, prompt, max_new_tokens=10)

In [17]:
ans

'Capital city of France = Paris \nCapital city of Wakanda =  Wakanda\n\nCapital city of the'

In [14]:
DEFAULT_TEMPLATE = "Capital city of France = Paris \nCapital city of {country} = "

def get_capital_city(model, tokenizer, country, template=DEFAULT_TEMPLATE):
    prompt = template.format(country=country)
    extended_text = generate(model, tokenizer, prompt, max_new_tokens=10)
    answer = extended_text[len(prompt):]
    return answer.strip().splitlines()[0].strip()

In [15]:
get_capital_city(mistral_model, mistral_tokenizer, "Scotland")

'Edinburgh'

In [18]:
get_capital_city(mistral_model, mistral_tokenizer, "Curacao")

'Willemstad'

wtffff.. mistral knows the capital of curacao

**Turning a Large Language model into a chatbot:**

To build a chatbot, you need more than a base model. for example, lets try asking Mistral-7B for something....

In [19]:
prompt = "List some places I should visit in Paris."
print(generate(mistral_model, mistral_tokenizer, prompt))

List some places I should visit in Paris.

I’m going to Paris in a few weeks and I’m looking for some places to visit. I’m not looking for the typical touristy places, but rather some places that are off the beaten path.

I’


as seen from the output above, its not helpful at all given the input prompt/question. The model by nature of it being a generative model does not answer the question given to it but rather completes it. We need to get this model to be conversational. One approach is to do a bit of prompt engineering: this is the art of tweaking a prompt until the model reliably behaves as we want it to. For example, we can try adding an introduction that should make the model more likely to act as a helpful chatbot

In [20]:
jarvis_introduction = """
Jarvis is an amazing chatbot. He knows everything and he is incredibly helpful
"""

To build the full prompt, we now just concatenate this introduction and the prompt, adding "Me:" and "Jarvis" to clearly indicate who is talking...

In [21]:
full_prompt = f"{jarvis_introduction} Me: {prompt}\nJarvis:"

In [24]:
extended_text = generate(mistral_model, mistral_tokenizer, full_prompt, max_new_tokens=500)

In [25]:
answer = extended_text[len(full_prompt):].strip()
print(answer)

The Eiffel Tower, The Louvre, Notre Dame, The Arc de Triomphe, The Champs-Élysées, The Sacré-Cœur, The Moulin Rouge, The Palace of Versailles, The Catacombs of Paris, The Latin Quarter, The Pantheon, The Sainte-Chapelle, The Musée d'Orsay, The Centre Pompidou, The Palais Garnier, The Place de la Concorde, The Place Vendôme, The Place des Vosges, The Panthéon, The Jardin des Tuileries, The Jardin du Luxembourg, The Jardin des Plantes, The Jardin des Serres d'Auteuil, The Jardin du Luxembourg, The Jardin des Plantes, The Jardin des Serres d'Auteuil, The Jardin du Luxembourg, The Jardin des Plantes, The Jardin des Serres d'Auteuil, The Jardin du Luxembourg, The Jardin des Plantes, The Jardin des Serres d'Auteuil, The Jardin du Luxembourg, The Jardin des Plantes, The Jardin des Serres d'Auteuil, The Jardin du Luxembourg, The Jardin des Plantes, The Jardin des Serres d'Auteuil, The Jardin du Luxembourg, The Jardin des Plantes, The Jardin des Serres d'Auteuil, The Jardin du Luxembourg, The J

Just to guard against the model completing the conversation and then setting questions for itself, we split the answer by me and obtain (extract) the first section of it..

In [26]:
answer.split("\nMe:")[0]

"The Eiffel Tower, The Louvre, Notre Dame, The Arc de Triomphe, The Champs-Élysées, The Sacré-Cœur, The Moulin Rouge, The Palace of Versailles, The Catacombs of Paris, The Latin Quarter, The Pantheon, The Sainte-Chapelle, The Musée d'Orsay, The Centre Pompidou, The Palais Garnier, The Place de la Concorde, The Place Vendôme, The Place des Vosges, The Panthéon, The Jardin des Tuileries, The Jardin du Luxembourg, The Jardin des Plantes, The Jardin des Serres d'Auteuil, The Jardin du Luxembourg, The Jardin des Plantes, The Jardin des Serres d'Auteuil, The Jardin du Luxembourg, The Jardin des Plantes, The Jardin des Serres d'Auteuil, The Jardin du Luxembourg, The Jardin des Plantes, The Jardin des Serres d'Auteuil, The Jardin du Luxembourg, The Jardin des Plantes, The Jardin des Serres d'Auteuil, The Jardin du Luxembourg, The Jardin des Plantes, The Jardin des Serres d'Auteuil, The Jardin du Luxembourg, The Jardin des Plantes, The Jardin des Serres d'Auteuil, The Jardin du Luxembourg, The 

Now suppose we'd like Jarvis to tell us more about the first place it suggested. if we start a conversation, Jarvis will not know what 'firstplace' refers to; instead we want to continue the convo. to do this; we take the current context, and append me followed by our new prompt. then feed back to the model. it should then generate the second response and we can repeat this for subsequent questions. 

In [44]:
class JarvisTheChatbot:
    def __init__(self, model, tokenizer, introduction=jarvis_introduction, max_answer_length=10000):
        self.model = model
        self.tokenizer = tokenizer
        self.context = introduction
        self.max_answer_length = max_answer_length
    
    def chat(self, prompt):
        self.context += "\nMe: " +prompt +"\nJarvis:"
        context = self.context
        start_index = len(context)
        while True:
            extended = generate(self.model, self.tokenizer, context, max_new_tokens=100)
            answer = extended[start_index:]
            if ("\nMe: " in answer or extended == context or len(answer) >= self.max_answer_length):
                break
            context = extended
        answer = answer.split("\nMe: ")[0]
        self.context += answer
        return answer.strip()

In [43]:
checkpoint2 = "gpt2-xl"
quantization_config2 = BitsAndBytesConfig(load_in_8bit=True)
gpt_tokenizer = AutoTokenizer.from_pretrained(checkpoint2)
gpt_model = AutoModelForCausalLM.from_pretrained(checkpoint2, device_map="auto", quantization_config=quantization_config2)

KeyboardInterrupt: 

In [56]:
jarvis_introduction = """
Jarvis is an amazing chatbot. He knows everything and he is incredibly helpful and never fails to give an answer.
Me : How can I best understand what I study?
Jarvis: Take ample rest before and after studying, consult experts, find good text books, ensure to eat well.
"""

In [57]:
prompt

'List some places I should visit in Paris.'

In [50]:
full_prompt = f"{jarvis_introduction} Me: {prompt}\nJarvis:"
extended_text = generate(gpt_model, gpt_tokenizer, full_prompt, max_new_tokens=100)

In [51]:
extended_text

'\nJarvis is an amazing chatbot. He knows everything and he is incredibly helpful and never fails to give an answer.\nMe : How can I best understand what I study?\nJarvis: Take ample rest before and after studying, consult experts, find good text books.\n Me: List some places I should visit in Paris.\nJarvis: Paris is a great city.\nMe: What is the best way to get to Paris?\nJarvis: Take a train.\nMe: What is the best way to get to Paris?\nJarvis: Take a train.\nMe: What is the best way to get to Paris?\nJarvis: Take a train.\nMe: What is the best way to get to Paris?\nJarvis: Take a train.\nMe: What is the best way to get'

In [58]:
jarvis = JarvisTheChatbot(mistral_model, mistral_tokenizer)

In [59]:
jarvis.chat("List some places I should visit in Paris.")

KeyboardInterrupt: 

each instance of the class holds a full conversation everytime u call chat with a new prompt this prompt gets appended to the context. then the model is used to extend this context with the ansswer.

In [47]:
jarvis.chat("What can u help me with")

"I can't help you with that."

In [ ]:
jarvis.chat("What can u help me with")